# Noise lab — every generation of the rotor-noise model, on one trajectory

One clip per **generation** of this project's rotor-noise model, all rendered on the
**same rotor-speed trajectory**, with the **same render seed**, under the **same level
rule** — so what is left between two clips is the noise model and nothing else.

| source | generation | what it is |
|---|---|---|
| `LegacyRandom(seed)` | legacy | a fresh draw of the hand-written family (`data_processing.stochastic_rotor_noise`), i.e. what `kind: stochastic` with a `ranges:` block renders per window |
| `LegacyBank("easy"/"hard", i)` | legacy | entry `i` of `data/rig_banks/rig_{easy,hard}_n2048.json` — the **exact** preset bank the `rig_easy` / `rig_hard` arms trained on |
| `V2Fit("dregon")` | v2 | the round-5 **uncalibrated** single-regime fit (`render.render_noise`) |
| `V2Fit("michaels")` | v2 | the round-3 standby + cruise pair (`render.render_noise_regimes`, smoothstep 45 → 65 rev/s) |
| `V2Bank("easy"/"hard", i)` | v2 | entry `i` of `data/rig_banks/noise_v2_{easy,hard}_n2048.json` (= `dload:noise-v2-banks`) |

The training policies these come from differ in four things at once — the level rule,
render/flight reuse, the speed scaling and the trajectory source — so two arms' clips
are never comparable as heard. Here **one** trajectory is drawn and **one** level rule
is applied to all of them.

Logic lives in `noise_lab.py`; the cells below are thin. There are no sliders: a knob is
a keyword argument (`LegacyRandom(seed=3, harm_mean_db=6.0)`).

Rendering only — no forward model is ever evaluated on a fit pool.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import noise_lab as NL

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110})

## 1. Pick a trajectory

`NL.trajectory(kind, ...)` returns a Frame with **two** rotor tracks: `rps` at 100 Hz
for the plots and `rps_render`, the exact 16 kHz carrier every renderer consumes.

* `"fitted"` — the fitted trajectory model (`dload:rps-traj-fits`, `measurement_noise=False`);
  `rig=` is any name of `NL.traj_rig_names()` or `"posterior"` (a fresh drone from the
  hyperprior), plus `mean_shift=`, `mean_scale=`, `full_flight=`.
* `"real"` — a recording's telemetry: `dataset=`, `recording=`, `offset_s=`.
* `"full_flight"` / `"intermittent"` / `"ou"` — the hand-written `rps_synthesis` scaffold.

`rps_scale=` multiplies the whole track exactly (a stopped rotor stays stopped).

In [ ]:
traj = NL.trajectory("fitted", rig="dregon", seed=0, duration_s=10.0)
NL.describe_traj(traj)

## 2. The sources

Seven of them: one legacy random draw, the two legacy bank entries the arms trained on,
the two winning v2 fits, and the two v2 bank entries.

In [ ]:
sources = [
    NL.LegacyRandom(seed=0),
    NL.LegacyBank("easy", 0),
    NL.LegacyBank("hard", 0),
    NL.V2Fit("dregon"),
    NL.V2Fit("michaels"),
    NL.V2Bank("easy", 0),
    NL.V2Bank("hard", 0),
]
for s in sources:
    print(f"{s.name:22s} {s.generation:7s} {s.entry}")

## 3. Render them all on that trajectory

`level=("window", 0.1)` is a per-clip RMS: it is the only way to hear the generations
against each other, because the v2 sources are in **absolute fitted units** while the
legacy model has an arbitrary internal scale. The native alternative is the last cell.

The span warnings are the point of the exercise, not noise: a fitted trajectory of a rig
routinely leaves the carrier range its noise fit was identified on, and there the fitted
speed envelopes extrapolate.

In [ ]:
frames = NL.render_all(sources, traj, seed=0, n_mics=1, level=("window", 0.1))
{k: tuple(v["audio"].data.shape) for k, v in frames.items()}

## 4. Look at them

Spectrogram over the rotor-speed track, one pair per source, colour clipped to the top
45 dB (the renders span ~145 dB, so an autoscale washes the comb out).

In [ ]:
NL.show(frames, dyn_range=45.0, figsize=(14, 26))

## 5. Hear them

Same trajectory, same seed, same level rule — so the difference is the model.

In [ ]:
NL.players(frames)

## 6. Read them

Round 4's order-tracked prominence over the local floor, in dB: the peak within ±1 bin
of `k·f_r` over the **median** of the two-sided 0.45–0.7·f̄ annulus, with every rotor's
`k−1 / k / k+1` lines excluded from the floor; 8192-point periodogram (1.95 Hz per bin)
at each clip's own carriers, mic-median then rotor-mean
(`results/noise_v2/rounds/round4/legacy_truth/anatomy.md`).

For reference, the same estimator on the round-4 free-flight window read **+6.2 dB** at
k=1 on the real recording, **+18.2 dB** on the legacy render and **+7.4 dB** on the v2
render.

In [ ]:
NL.line_stats(frames, k_max=8).round(2)

## 7. Model against realisation, for one v2 source

The fit's expected periodogram against the clip's realised one, in the absolute units the
fit is stated in — the notebook's level gain is divided out first. Legacy sources have no
comparable forward model here and are refused by name.

In [ ]:
NL.expected_vs_realised(sources[3], frames["v2-fit dregon"]);

## 8. The other level rule

`("flight", rms)` is the training policy's alternative: the number is the level **at the
reference speed**, and the clip's own floor envelope (`mean_r(speed**floor_exp) +
floor_static_rel`, the very factor each renderer shaped its floor with) is put back — so
a slow passage stays quieter than a fast one instead of every window leaving at the same
level. Each source reports its own envelope, so the rule means the same thing across
generations.

`level=None` keeps each generation's native scale: the fit's own absolute RMS for a v2
source, the model's arbitrary scale for a legacy one.

In [ ]:
flight = NL.render(sources[3], traj, seed=0, level=("flight", 0.1))
native = NL.render(sources[3], traj, seed=0, level=None)
for tag, f in (("window", frames["v2-fit dregon"]), ("flight", flight), ("native", native)):
    m = f["meta"]
    print(f"{tag:7s} rms {float(m['rms']):.4g}   gain {float(m['level_gain']):.4g}")
NL.players({"v2-fit dregon, flight level": flight})